# 02 — DistilBERT Fine-tuning (SST-2)

**Branch:** `reproduction/distilbert`  
**Owner:** Person C

Fine-tunes `distilbert-base-uncased` on SST-2 using the same setup as notebook 01.
Results are compared against BERT-base to verify the paper's claims.

**Expected results (from paper):** ~91.3% accuracy — retaining 97% of BERT-base performance.

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

In [ ]:
import sys
sys.path.append('../src')

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

from train import train_model
from evaluate import evaluate_model
from utils import print_model_info, measure_inference_speed, save_results, count_parameters, model_size_mb

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# ── Config — identical to notebook 01 for fair comparison ─────────────────
MODEL_NAME   = 'distilbert-base-uncased'
MAX_LENGTH   = 128
BATCH_SIZE   = 32
EPOCHS       = 3
LR           = 2e-5
SEED         = 42

torch.manual_seed(SEED)

In [ ]:
# ── Load & tokenize SST-2 ──────────────────────────────────────────────────
# Note: DistilBERT shares the same tokenizer as BERT-base
raw = load_dataset('glue', 'sst2')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['sentence'], truncation=True, max_length=MAX_LENGTH, padding='max_length')

tokenized = raw.map(tokenize, batched=True)

# DistilBERT has no token_type_ids — remove them if present
cols = ['input_ids', 'attention_mask', 'label']
tokenized.set_format('torch', columns=cols)
tokenized = tokenized.rename_column('label', 'labels')

train_loader = DataLoader(tokenized['train'],     batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(tokenized['validation'], batch_size=BATCH_SIZE)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
# ── Load model ─────────────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
print_model_info(model, 'DistilBERT')

In [ ]:
history = train_model(model, train_loader, val_loader, DEVICE, epochs=EPOCHS, lr=LR)

In [ ]:
eval_results = evaluate_model(model, val_loader, DEVICE)
print(f"Validation accuracy: {eval_results['accuracy']}%")

In [ ]:
speed = measure_inference_speed(model, val_loader, DEVICE)
print(speed)

In [ ]:
total_params, _ = count_parameters(model)

results = {
    'model':             MODEL_NAME,
    'task':              'sst2',
    'val_accuracy':      eval_results['accuracy'],
    'total_parameters':  total_params,
    'model_size_mb':     model_size_mb(model),
    'inference_speed':   speed,
    'training_history':  history,
}

save_results(results, 'distilbert.json')
print(results)